# Index Search

Scan multiple index universes in one run and show one combined ranked result.


In [ ]:
import analysis_interfaces.interface_index_search as iss
import pandas as pd


## Block 1: Parameters


In [2]:
#index_names = ["dow30", "nasdaq100", "sp500"]
#limits_by_index = {
#    "dow30": 30,
#    "nasdaq100": 100,
#    "sp500": 500,
#}
index_names = ["dow30", "nasdaq100"]
limits_by_index = {
    "dow30": 30,
    "nasdaq100": 100
}
include_sentiment = False  # True requires API keys and adds latency
ticker_cache_dir = "cache"
ticker_cache_max_age_hours = 24


## Block 2: Run All Index Scans


In [ ]:
results_by_index = {}
prediction_frames = []
ticker_cache_rows = []

for index_name in index_names:
    result = iss.run_index_search_workflow(
        index_name=index_name,
        limit=limits_by_index.get(index_name),
        include_sentiment=include_sentiment,
        use_ticker_cache=True,
        ticker_cache_dir=ticker_cache_dir,
        ticker_cache_max_age_hours=ticker_cache_max_age_hours,
    )

    results_by_index[index_name] = result

    prediction_frame = result["prediction_summary"].copy()
    if not prediction_frame.empty:
        prediction_frame.insert(0, "index_name", index_name)
        prediction_frames.append(prediction_frame)

    ticker_cache = result.get("ticker_cache") or {}
    ticker_cache_rows.append({
        "index_name": index_name,
        "tickers_scanned": len(result.get("tickers", [])),
        "cache_used": ticker_cache.get("cache_used"),
        "cache_file": ticker_cache.get("cache_file"),
        "cache_age_hours": ticker_cache.get("cache_age_hours"),
    })

combined_prediction_summary = (
    pd.concat(prediction_frames, ignore_index=True)
    if prediction_frames
    else pd.DataFrame()
)

ticker_cache_summary = pd.DataFrame(ticker_cache_rows)


## Block 3: Cache Summary


In [4]:
ticker_cache_summary


,index_name,tickers_scanned,cache_used,cache_file,cache_age_hours
0,dow30,30,True,cache\dow30_tickers.json,12.196
1,nasdaq100,100,True,cache\nasdaq100_tickers.json,12.202


## Block 4: Combined Ranked Output


In [5]:
ranked = combined_prediction_summary.sort_values(
    by=["Signal", "index_name", "TICKER"],
    ascending=[False, True, True],
).reset_index(drop=True)

signal_only = ranked[ranked["Signal_Text"] != "HOLD"].reset_index(drop=True)

signal_only


,index_name,current_date,TICKER,Signal,Signal_Text
0,nasdaq100,20260315,PDD,2,STRONG BUY
1,dow30,20260315,JPM,1,WEAK BUY
2,dow30,20260315,UNH,1,WEAK BUY
3,dow30,20260315,VZ,1,WEAK BUY
4,nasdaq100,20260315,BKNG,1,WEAK BUY
5,nasdaq100,20260315,CHTR,1,WEAK BUY
6,nasdaq100,20260315,CMCSA,1,WEAK BUY
7,nasdaq100,20260315,TMUS,1,WEAK BUY
